# Xử lý & Làm sạch dữ liệu `patient.csv` (eICU Demo v2.0.1)

**Mục tiêu:** Chuẩn bị dữ liệu đầu vào cho bài toán dự đoán tử vong ICU (ICU Mortality Prediction).

**Input:** `data/raw/eicu/patient.csv`
**Output:** `data/processed/patient_clean.csv`

**Quy trình:** Nạp dữ liệu → Chọn cột → Chuẩn hóa tuổi → Lọc cohort → Feature Engineering → Kiểm tra & Lưu.

In [31]:
# ============================================================
# BƯỚC 1: KHỞI TẠO VÀ NẠP DỮ LIỆU GỐC
# ============================================================
# Mục đích: Đọc file patient.csv thô từ eICU Demo.
# Bảng patient.csv chứa thông tin nhân khẩu học và trạng thái
# xuất viện của từng đợt nằm ICU — đây là nguồn cung cấp NHÃN
# (mortality) quan trọng nhất cho mô hình.

import pandas as pd
import numpy as np
import os

# Đường dẫn file vào / ra
INPUT_PATH = "../data/raw/eicu/patient.csv"
OUTPUT_PATH = "../data/processed/patient_clean.csv"

# Đọc dữ liệu gốc
df = pd.read_csv(INPUT_PATH)

# Kiểm tra kích thước tổng thể: (số dòng, số cột)
print("Kích thước dữ liệu gốc:", df.shape)

Kích thước dữ liệu gốc: (2520, 29)


In [32]:
# ============================================================
# BƯỚC 2: CHỌN CỘT VÀ KIỂM TRA DỮ LIỆU THÔ
# ============================================================
# Giữ 7 cột cần thiết để tối ưu bộ nhớ và tránh nhiễu:
#   1. patientunitstayid     : Mã đợt nằm ICU (khóa chính, dùng để JOIN)
#   2. age                   : Tuổi bệnh nhân (dạng string, cần xử lý)
#   3. gender                : Giới tính (M/F/U)
#   4. admissionweight       : Cân nặng lúc nhập viện (kg)
#   5. admissionheight       : Chiều cao lúc nhập viện (cm)
#   6. unitdischargeoffset   : Thời gian nằm ICU (phút)
#   7. hospitaldischargestatus: Trạng thái ra viện (Alive/Expired) -> NHÃN

COLS_NEEDED = [
    "patientunitstayid",
    "age",
    "gender",
    "admissionweight",
    "admissionheight",
    "unitdischargeoffset",
    "hospitaldischargestatus",
]

df = df[COLS_NEEDED].copy()

print("Kích thước sau khi chọn cột:", df.shape)
print("\nThông tin kiểu dữ liệu:")
df.info()

print("\n5 dòng đầu tiên:")
display(df.head())

print("\nGiá trị duy nhất của 'gender':", df["gender"].unique())
print("Giá trị duy nhất của 'hospitaldischargestatus':", df["hospitaldischargestatus"].unique())

Kích thước sau khi chọn cột: (2520, 7)

Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patientunitstayid        2520 non-null   int64  
 1   age                      2516 non-null   str    
 2   gender                   2516 non-null   str    
 3   admissionweight          2322 non-null   float64
 4   admissionheight          2451 non-null   float64
 5   unitdischargeoffset      2520 non-null   int64  
 6   hospitaldischargestatus  2492 non-null   str    
dtypes: float64(2), int64(2), str(3)
memory usage: 168.3 KB

5 dòng đầu tiên:


,patientunitstayid,age,gender,admissionweight,admissionheight,unitdischargeoffset,hospitaldischargestatus
0,141764,87,Female,NaN,157.5,344,Alive
1,141765,87,Female,46.5,157.5,2250,Alive
2,143870,76,Male,77.5,167.0,793,Alive
3,144815,34,Female,60.3,172.7,1121,Alive
4,145427,61,Male,91.7,177.8,1369,Alive



Giá trị duy nhất của 'gender': <ArrowStringArray>
['Female', 'Male', nan]
Length: 3, dtype: str
Giá trị duy nhất của 'hospitaldischargestatus': <ArrowStringArray>
['Alive', 'Expired', nan]
Length: 3, dtype: str


In [33]:
# ============================================================
# BƯỚC 3: XỬ LÝ & CHUẨN HÓA CỘT TUỔI (age)
# ============================================================
# VẤN ĐỀ: Trong eICU, cột age lưu dạng STRING và chứa giá trị đặc biệt "> 89" (do quy định bảo mật thông tin người cao tuổi —HIPAA). Nếu không xử lý, pd.to_numeric sẽ biến chúng thành NaN và ta mất dữ liệu của nhóm bệnh nhân cao tuổi nhất — vốn có nguy cơ tử vong cao nhất.
# GIẢI PHÁP:
#   1. Thay "> 89" thành 90 (giá trị đại diện hợp lý).
#   2. Ép toàn bộ cột sang kiểu số bằng pd.to_numeric(errors="coerce").
#   3. Kiểm tra NaN phát sinh.

print("Giá trị age độc nhất (mẫu):", df["age"].unique())

# 1. Thay thế "> 89" -> 90
df["age"] = df["age"].replace("> 89", "90")

# 2. Ép kiểu số; các giá trị không hợp lệ -> NaN
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 3. Kiểm tra NaN sau ép kiểu
n_age_nan = df["age"].isna().sum()
print(f"\nSố dòng age bị NaN sau ép kiểu: {n_age_nan}")

print("\nThống kê mô tả cột age:")
df["age"].describe()

Giá trị age độc nhất (mẫu): <ArrowStringArray>
[  '87',   '76',   '34',   '61',   '55',   '60',   '28', '> 89',   '59',
   '44',   '66',   '41',   '63',   '57',   '52',   '23',   '73',   '39',
   '20',   '29',   '71',   '18',   '83',   '84',   '47',   '38',   '46',
   '49',   '81',   '68',   '51',   '17',   '89',   '85',   '56',   '80',
   '48',   '74',   '16',   '64',   '75',   '67',   '72',   '82',   '77',
   '78',   '58',   '88',   '53',   '54',   '65',   '69',   '62',   '50',
   '42',   '32',   '25',   '40',   '43',   '79',   '70',   '86',   '30',
   '37',   '26',   '19',   '31',   '21',   '45',    nan,   '22',   '27',
   '33',   '35',   '24',   '36',   '15']
Length: 77, dtype: str

Số dòng age bị NaN sau ép kiểu: 4

Thống kê mô tả cột age:


count    2516.000000
mean       63.281002
std        17.764847
min        15.000000
25%        53.000000
50%        66.000000
75%        77.000000
max        90.000000
Name: age, dtype: float64

In [34]:
# ============================================================
# BƯỚC 4: LỌC MẪU BỆNH NHÂN (COHORT FILTERING)
# ============================================================
# Áp dụng 3 tiêu chuẩn lọc theo phương pháp nghiên cứu lâm sàng:
#
#   (1) age >= 18:
#       Loại bỏ nhi khoa (PICU/NICU) vì sinh lý học trẻ em
#       hoàn toàn khác người lớn -> không thể dùng chung mô hình.
#
#   (2) unitdischargeoffset >= 240 phút (>= 4 giờ):
#       Loại bỏ các ca chuyển viện khẩn cấp hoặc tử vong quá nhanh,
#       không đủ cửa sổ 24h để thu thập dữ liệu sinh hiệu/xét nghiệm.
#       (Đây là tiêu chí bài báo gốc sử dụng.)
#
#   (3) Loại bỏ dòng thiếu nhãn hospitaldischargestatus:
#       Không có nhãn -> không thể huấn luyện có giám sát.

n_before = len(df)
n_age_chua_18 = len(df[df["age"] < 18])
n_unit_be_hon_4_tieng = len(df[df["unitdischargeoffset"] < 240])

# (1) Lọc tuổi >= 18
df = df[df["age"] >= 18]

# (2) Lọc thời gian nằm ICU >= 240 phút (~4 giờ)
df = df[df["unitdischargeoffset"] >= 240]

# (3) Loại bỏ dòng thiếu nhãn
df = df.dropna(subset=["hospitaldischargestatus"])

n_after = len(df)
pct_dropped = (1 - n_after / n_before) * 100

print(f"Số dòng trước lọc : {n_before:,}")
print(f"Số dòng sau lọc   : {n_after:,}")
print(f"Đã loại bỏ        : {n_before - n_after:,} dòng ({pct_dropped:.2f}%)")
print("Số bệnh nhân <18 tuổi: ", n_age_chua_18)
print("Số bệnh nhân trong ICU ko quá 4 tiếng: ", n_unit_be_hon_4_tieng)

# ---------- (4) PLAUSIBILITY FILTERING cho weight/height ----------
# Đếm số dòng bất thường TRƯỚC khi gán NaN (để báo cáo)
mask_w_bad = df["admissionweight"].notna() & ~df["admissionweight"].between(20, 300)
mask_h_bad = df["admissionheight"].notna() & ~df["admissionheight"].between(100, 250)

print(f"\n[Plausibility] Cân nặng ngoài [20, 300] kg : {mask_w_bad.sum()} dòng")
print(f"[Plausibility] Chiều cao ngoài [100, 250] cm: {mask_h_bad.sum()} dòng")

# Gán NaN cho các giá trị phi lý (KHÔNG xóa dòng)
df.loc[mask_w_bad, "admissionweight"] = np.nan
df.loc[mask_h_bad, "admissionheight"] = np.nan

# Kiểm tra lại sau khi lọc hợp lý
print("\nThống kê weight sau plausibility filter:")
print(df["admissionweight"].describe())
print("\nThống kê height sau plausibility filter:")
print(df["admissionheight"].describe())

# ---------- Kiểm tra nhãn ----------
print("\nPhân bố nhãn sau lọc cohort:")
print(df["hospitaldischargestatus"].value_counts())

Số dòng trước lọc : 2,520
Số dòng sau lọc   : 2,326
Đã loại bỏ        : 194 dòng (7.70%)
Số bệnh nhân <18 tuổi:  4
Số bệnh nhân trong ICU ko quá 4 tiếng:  160

[Plausibility] Cân nặng ngoài [20, 300] kg : 2 dòng
[Plausibility] Chiều cao ngoài [100, 250] cm: 12 dòng

Thống kê weight sau plausibility filter:
count    2189.000000
mean       83.166994
std        25.103865
min        32.700000
25%        65.800000
50%        79.370000
75%        96.880000
max       293.930000
Name: admissionweight, dtype: float64

Thống kê height sau plausibility filter:
count    2268.000000
mean      170.185904
std        10.835462
min       109.700000
25%       162.600000
50%       170.200000
75%       177.800000
max       204.500000
Name: admissionheight, dtype: float64

Phân bố nhãn sau lọc cohort:
hospitaldischargestatus
Alive      2135
Expired     191
Name: count, dtype: int64


In [35]:
# ============================================================
# BƯỚC 5: BIẾN ĐỔI VÀ MÃ HÓA ĐẶC TRƯNG (FEATURE ENGINEERING)
# ============================================================

# ---------- 5.1: Tính chỉ số BMI ----------
# Công thức: BMI = weight(kg) / (height(m))^2
# Lưu ý: weight/height đã được làm sạch bởi plausibility filter ở BƯỚC trước,
# nên ở đây chỉ cần tính khi cả hai đều không NaN.

height_m = df["admissionheight"] / 100.0

valid_mask = (
    df["admissionweight"].notna() & (df["admissionweight"] > 0) &
    height_m.notna() & (height_m > 0)
)

df["bmi"] = np.nan
df.loc[valid_mask, "bmi"] = (
    df.loc[valid_mask, "admissionweight"] / (height_m[valid_mask] ** 2)
)

# ---------- 5.1b: BMI PLAUSIBILITY FILTER ----------
# Theo contracts: chỉ giữ 10 <= BMI <= 80. Ngoài khoảng -> NaN.
# Đây là lớp bảo vệ thứ 2, phòng trường hợp weight/height đã hợp lệ
# nhưng kết hợp lại vẫn cho BMI bất thường (hiếm nhưng có thể xảy ra).
mask_bmi_bad = df["bmi"].notna() & ~df["bmi"].between(10, 80)
print(f"[Plausibility] BMI ngoài [10, 80]: {mask_bmi_bad.sum()} dòng")
df.loc[mask_bmi_bad, "bmi"] = np.nan

print("\nThống kê BMI sau làm sạch:")
print(df["bmi"].describe())
print(f"Số dòng BMI NaN: {df['bmi'].isna().sum():,} "
      f"({df['bmi'].isna().mean()*100:.2f}%)")

# ---------- 5.2: Mã hóa giới tính -> male_sex ----------
# 1 = Male, 0 = Female, NaN = Unknown (U)
df["male_sex"] = df["gender"].map({"Male": 1, "Female": 0})

print("\nPhân bố male_sex:")
print(df["male_sex"].value_counts(dropna=False))

# ---------- 5.3: Mã hóa nhãn -> mortality ----------
# 1 = Expired (tử vong), 0 = Alive (sống sót)
df["mortality"] = df["hospitaldischargestatus"].map(
    {"Expired": 1, "Alive": 0}
)

print("\nPhân bố nhãn mortality:")
print(df["mortality"].value_counts(dropna=False))

assert df["mortality"].isna().sum() == 0, "Còn NaN ở nhãn mortality!"

[Plausibility] BMI ngoài [10, 80]: 1 dòng

Thống kê BMI sau làm sạch:
count    2157.000000
mean       28.573830
std         8.023008
min        12.549751
25%        23.077560
50%        27.256803
75%        32.182546
max        79.978849
Name: bmi, dtype: float64
Số dòng BMI NaN: 169 (7.27%)

Phân bố male_sex:
male_sex
1.0    1402
0.0     922
NaN       2
Name: count, dtype: int64

Phân bố nhãn mortality:
mortality
0    2135
1     191
Name: count, dtype: int64


In [36]:
# ============================================================
# BƯỚC 6: KIỂM TRA CUỐI CÙNG VÀ LƯU FILE
# ============================================================

# Chỉ giữ 5 cột sạch chuẩn bị cho mô hình
FINAL_COLS = ["patientunitstayid", "age", "male_sex", "bmi", "mortality"]
df_final = df[FINAL_COLS].copy()

# ---------- Kiểm tra tính duy nhất của khóa chính ----------
n_dup = df_final["patientunitstayid"].duplicated().sum()
print(f"Số patientunitstayid bị trùng lặp: {n_dup}")
assert n_dup == 0, "Có patientunitstayid bị trùng — cần kiểm tra lại!"

print(df_final.info())

# ---------- Thống kê mô tả ----------
print("\nThống kê mô tả dữ liệu cuối:")
print(df_final.describe(include="all"))

# ---------- Tỷ lệ mất cân bằng nhãn ----------
print("\nTỷ lệ nhãn mortality:")
print(df_final["mortality"].value_counts(normalize=True).round(4))

# ---------- Lưu file ----------
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Đã lưu file: {OUTPUT_PATH}")
print(f"   Kích thước cuối: {df_final.shape}")

Số patientunitstayid bị trùng lặp: 0
<class 'pandas.DataFrame'>
Index: 2326 entries, 0 to 2519
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patientunitstayid  2326 non-null   int64  
 1   age                2326 non-null   float64
 2   male_sex           2324 non-null   float64
 3   bmi                2157 non-null   float64
 4   mortality          2326 non-null   int64  
dtypes: float64(3), int64(2)
memory usage: 109.0 KB
None

Thống kê mô tả dữ liệu cuối:
       patientunitstayid          age     male_sex          bmi    mortality
count       2.326000e+03  2326.000000  2324.000000  2157.000000  2326.000000
mean        1.698311e+06    63.370593     0.603270    28.573830     0.082115
std         1.017670e+06    17.675091     0.489324     8.023008     0.274599
min         1.417640e+05    18.000000     0.000000    12.549751     0.000000
25%         8.270852e+05    53.000000     0.000000    23.077560  

## 📊 Tóm tắt quy trình

| Bước | Mô tả | Kết quả |
|------|-------|---------|
| 1 | Nạp dữ liệu gốc | `patient.csv` |
| 2 | Chọn 7 cột cần thiết | Giảm bộ nhớ |
| 3 | Xử lý `age` (`"> 89"` → `90`) | Cột số hợp lệ |
| 4 | Lọc cohort (age≥18, LOS≥240p, có nhãn) | Cohort người lớn ICU |
| 5 | Tính BMI, mã hóa `male_sex`, `mortality` | 3 đặc trưng mới |
| 6 | Kiểm tra & lưu | `patient_clean.csv` |

**Output cuối:** `patientunitstayid`, `age`, `male_sex`, `bmi`, `mortality`